# 11 — Deep Learning: LSTM / GRU RW-Correction

This notebook tests compact LSTM and GRU models as Random-Walk correction models for USD, GBP, EUR and JPY at H1, H3 and H7.

Rules: temporal validation only; training targets must resolve before validation starts; scaling fitted on training only; 2024–2025 remains untouched; Script 09 is retained as fallback.

In [ ]:
from pathlib import Path
import json, os, random, warnings, time
import numpy as np
import pandas as pd
from scipy.stats import t as student_t
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
os.environ["TF_CPP_MIN_LOG_LEVEL"]="2"
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input,LSTM,GRU,Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
warnings.filterwarnings("ignore")
SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
CURRENCIES=["USD","GBP","EUR","JPY"]; HORIZONS=[1,3,7]; YEARS=[2019,2020,2021,2022,2023]
SEQS=[7,14,30]; FAMILIES=["TT_OWN","TT_CROSS"]; MODELS=["LSTM","GRU"]
SHRINK=[0.0,0.1,0.25,0.5,0.75,1.0]; BLEND=[0.0,0.25,0.5,0.75,1.0]
EPOCHS=40; BATCH=32; PATIENCE=5
print("TensorFlow",tf.__version__)

In [ ]:
def find_root():
    for root in [Path.cwd(),Path.cwd().parent,Path.cwd().parent.parent]:
        if (root/"data/processed/02_development_model_dataset.csv").exists() and (root/"outputs/predictions/09_online_hybrid_predictions_2019_2023.csv").exists(): return root.resolve()
    raise FileNotFoundError("Required project files not found")
ROOT=find_root(); PRED=ROOT/"outputs/predictions"; TAB=ROOT/"outputs/tables"; TAB.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(ROOT/"data/processed/02_development_model_dataset.csv",parse_dates=["Date"])
for h in HORIZONS: df[f"Target_Date_H{h}"]=pd.to_datetime(df[f"Target_Date_H{h}"],errors="coerce")
with open(ROOT/"data/processed/02_feature_sets.json",encoding="utf-8") as f: FEATURE_SETS=json.load(f)
rw=pd.read_csv(PRED/"04_random_walk_predictions_2019_2023.csv",parse_dates=["Forecast_Origin","Target_Date"])
s09=pd.read_csv(PRED/"09_online_hybrid_predictions_2019_2023.csv",parse_dates=["Forecast_Origin","Target_Date"])
assert df["Date"].max()<pd.Timestamp("2024-01-01")
print("PASS — final holdout untouched")

In [ ]:
def rmse(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p); return float(np.sqrt(np.mean((y[m]-p[m])**2))) if m.sum() else np.nan
def mae(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p); return float(np.mean(np.abs(y[m]-p[m]))) if m.sum() else np.nan
def skill(m,r): return float(1-m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan
def theils_u(m,r): return float(m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan
def dm_test(y,model,rw_pred,horizon=1):
    y=np.asarray(y,float); model=np.asarray(model,float); rw_pred=np.asarray(rw_pred,float)
    mask=np.isfinite(y)&np.isfinite(model)&np.isfinite(rw_pred); y=y[mask]; model=model[mask]; rw_pred=rw_pred[mask]; n=len(y)
    if n<20: return np.nan,np.nan
    d=(y-rw_pred)**2-(y-model)**2; lag=max(0,int(horizon)-1); lrv=np.var(d,ddof=1)
    for k in range(1,lag+1):
        if len(d)<=k: break
        lrv += 2*(1-k/(lag+1))*np.cov(d[k:],d[:-k],ddof=1)[0,1]
    if not np.isfinite(lrv) or lrv<=0: return np.nan,np.nan
    stat=np.mean(d)/np.sqrt(lrv/n); p=2*student_t.sf(abs(stat),df=n-1); return float(stat),float(p)

In [ ]:
def get_features(cur, family):

    raw = FEATURE_SETS[cur][family]

    # Remove duplicates while preserving order
    feats = list(dict.fromkeys(raw))

    # Only keep columns that really exist
    feats = [
        c for c in feats
        if c in df.columns
    ]

    # These columns are handled separately and
    # must NOT appear again inside the feature list.
    reserved = {
        "Date",
        f"{cur}_mid",
        f"{cur}_change_H1",
        f"{cur}_change_H3",
        f"{cur}_change_H7",
        "Target_Date_H1",
        "Target_Date_H3",
        "Target_Date_H7",
    }

    feats = [
        c for c in feats
        if c not in reserved
    ]

    return feats

In [ ]:
ERROR_LOG=[]

def tune(cur,h,outer_year):
    inner=outer_year-1
    rows=[]

    for fam in FAMILIES:
        for seq in SEQS:
            for kind in MODELS:
                try:
                    r=fitpred(cur,h,fam,seq,kind,inner)
                except Exception as e:
                    ERROR_LOG.append({
                        "Currency":cur,
                        "Horizon":h,
                        "Outer_Year":outer_year,
                        "Inner_Year":inner,
                        "Family":fam,
                        "Sequence":seq,
                        "Model":kind,
                        "Error":repr(e)
                    })
                    continue

                actual=r["Mid"]+r["ActualChange"]
                rwv=r["Mid"]

                for lam in SHRINK:
                    pred=rwv+lam*(r["RawLevel"]-rwv)
                    rows.append({
                        "Feature_Family":fam,
                        "Sequence_Length":seq,
                        "Model_Type":kind,
                        "Lambda":lam,
                        "Inner_RMSE":rmse(actual,pred),
                        "Feature_Count":r["Feature_Count"]
                    })

    tab=pd.DataFrame(rows)

    if tab.empty:
        return None,tab

    b=tab.sort_values(["Inner_RMSE","Sequence_Length"]).iloc[0]

    return {
        "Feature_Family":b["Feature_Family"],
        "Sequence_Length":int(b["Sequence_Length"]),
        "Model_Type":b["Model_Type"],
        "Lambda":float(b["Lambda"]),
        "Inner_RMSE":float(b["Inner_RMSE"]),
        "Feature_Count":int(b["Feature_Count"])
    },tab

def outer_pred(cur,h,year,sel):
    r=fitpred(cur,h,sel["Feature_Family"],sel["Sequence_Length"],sel["Model_Type"],year)

    dl=r["Mid"]+sel["Lambda"]*(r["RawLevel"]-r["Mid"])

    out=pd.DataFrame({
        "Currency":cur,
        "Horizon":h,
        "Validation_Year":year,
        "Forecast_Origin":r["Dates"],
        "DL_Prediction":dl
    })

    ex=rw[(rw.Currency==cur)&(rw.Horizon==h)&(rw.Forecast_Origin.dt.year==year)][
        ["Forecast_Origin","Target_Date","Actual","RW_Prediction"]
    ].copy()

    col="Online_Hybrid_Prediction" if "Online_Hybrid_Prediction" in s09.columns else "Prediction"

    ex=ex.merge(
        s09[(s09.Currency==cur)&(s09.Horizon==h)&(s09.Forecast_Origin.dt.year==year)][
            ["Forecast_Origin",col]
        ].rename(columns={col:"Script09_Prediction"}),
        on="Forecast_Origin",how="left"
    )

    return ex.merge(out,on="Forecast_Origin",how="inner")

def choose_blend(prev):
    if prev is None or len(prev)==0:
        return 0.0,np.nan

    rows=[]
    y=prev.Actual.to_numpy(float)
    p09=prev.Script09_Prediction.to_numpy(float)
    pdl=prev.DL_Prediction.to_numpy(float)

    for a in BLEND:
        rows.append({
            "Alpha_DL":a,
            "RMSE":rmse(y,(1-a)*p09+a*pdl)
        })

    b=pd.DataFrame(rows).sort_values("RMSE").iloc[0]
    return float(b.Alpha_DL),float(b.RMSE)


In [ ]:
# ============================================================
# PRE-FLIGHT DIAGNOSTIC — FIXED
# ============================================================

for cur in CURRENCIES:

    for fam in FAMILIES:

        feats = get_features(
            cur,
            fam
        )

        usable = []

        for c in feats:

            value = df.loc[:, c]

            # If duplicate column names exist,
            # take the first matching physical column.
            if isinstance(value, pd.DataFrame):
                value = value.iloc[:, 0]

            s = pd.to_numeric(
                value,
                errors="coerce"
            )

            if s.notna().sum() >= 20:
                usable.append(c)

        print(
            f"{cur} {fam}: "
            f"candidates={len(feats)}, "
            f"usable_numeric={len(usable)}"
        )


# ------------------------------------------------------------
# DIRECT SEQUENCE TEST
# ------------------------------------------------------------

_test = prep(
    "USD",
    1,
    FAMILIES[0],
    7,
    2018
)

print(
    "\nUSD H1 / 2018 sequence test:",
    "train =", len(_test[0]),
    "| validation =", len(_test[2]),
    "| features =", len(_test[-1])
)

In [ ]:
allout=[]; sels=[]; cands=[]

for cur in CURRENCIES:
    for h in HORIZONS:
        print(f"\n===== {cur} H{h} =====",flush=True)
        prev=None

        for year in YEARS:
            t0=time.time()
            sel,tab=tune(cur,h,year)

            if sel is None:
                print(year,"NO VALID DL CONFIG")
                recent=[x for x in ERROR_LOG if x["Currency"]==cur and x["Horizon"]==h and x["Outer_Year"]==year]
                if recent:
                    print("Example error:",recent[0]["Error"])
                continue

            try:
                outy=outer_pred(cur,h,year,sel)
            except Exception as e:
                print(year,"outer prediction failed:",repr(e))
                continue

            if outy.empty:
                print(year,"no matched forecast origins")
                continue

            a,br=(0.0,np.nan) if year==YEARS[0] else choose_blend(prev)

            outy["Alpha_DL"]=a
            outy["Final_DL_Hybrid_Prediction"]=(1-a)*outy["Script09_Prediction"]+a*outy["DL_Prediction"]

            allout.append(outy)
            sels.append({
                "Currency":cur,
                "Horizon":h,
                "Outer_Year":year,
                **sel,
                "Alpha_DL":a,
                "Blend_Inner_RMSE":br
            })

            if not tab.empty:
                tab=tab.copy()
                tab["Currency"]=cur
                tab["Horizon"]=h
                tab["Outer_Year"]=year
                cands.append(tab)

            prev=outy.copy()

            print(
                f"{year} | {sel['Model_Type']} seq={sel['Sequence_Length']} "
                f"{sel['Feature_Family']} features={sel['Feature_Count']} "
                f"lambda={sel['Lambda']:.2f} DLblend={a:.2f} "
                f"| {time.time()-t0:.1f}s",
                flush=True
            )

if not allout:
    err_df=pd.DataFrame(ERROR_LOG)
    display(err_df.head(20))
    raise RuntimeError(
        "No DL forecasts were produced. The table above now shows the real underlying errors."
    )

final_predictions=pd.concat(allout,ignore_index=True)
selections=pd.DataFrame(sels)
print("Successful forecast rows:",len(final_predictions))


In [ ]:
pooled=[]
for (cur,h),g in final_predictions.groupby(["Currency","Horizon"]):
    rr=rmse(g.Actual,g.RW_Prediction); r09=rmse(g.Actual,g.Script09_Prediction); rdl=rmse(g.Actual,g.DL_Prediction); rf=rmse(g.Actual,g.Final_DL_Hybrid_Prediction); _,p=dm_test(g.Actual,g.Final_DL_Hybrid_Prediction,g.RW_Prediction,h)
    pooled.append({"Currency":cur,"Horizon":h,"Observations":len(g),"RW_RMSE":rr,"Script09_RMSE":r09,"Raw_DL_RMSE":rdl,"Final_Hybrid_RMSE":rf,"Script09_Skill":skill(r09,rr),"Final_Hybrid_Skill":skill(rf,rr),"Final_Theils_U":theils_u(rf,rr),"DM_pvalue_vs_RW":p,"Beats_RW":rf<rr})
pooled_results=pd.DataFrame(pooled).sort_values(["Currency","Horizon"]); display(pooled_results); print("Tasks beating RW:",int(pooled_results.Beats_RW.sum()),"/",len(pooled_results))

In [ ]:
yearly=[]
for (cur,h,year),g in final_predictions.groupby(["Currency","Horizon","Validation_Year"]):
    rr=rmse(g.Actual,g.RW_Prediction); mr=rmse(g.Actual,g.Final_DL_Hybrid_Prediction); _,p=dm_test(g.Actual,g.Final_DL_Hybrid_Prediction,g.RW_Prediction,h)
    yearly.append({"Currency":cur,"Horizon":h,"Validation_Year":year,"Observations":len(g),"RMSE":mr,"MAE":mae(g.Actual,g.Final_DL_Hybrid_Prediction),"Forecast_Skill_vs_RW":skill(mr,rr),"Theils_U":theils_u(mr,rr),"DM_pvalue_vs_RW":p})
yearly_results=pd.DataFrame(yearly).sort_values(["Currency","Horizon","Validation_Year"]); display(yearly_results)

In [ ]:
final_predictions.to_csv(PRED/"11_deep_learning_predictions_2019_2023.csv",index=False); pooled_results.to_csv(TAB/"11_deep_learning_pooled_results.csv",index=False); yearly_results.to_csv(TAB/"11_deep_learning_yearly_results.csv",index=False); selections.to_csv(TAB/"11_deep_learning_selections.csv",index=False)
if cands: pd.concat(cands,ignore_index=True).to_csv(TAB/"11_deep_learning_candidates.csv",index=False)
quality_gate=pd.DataFrame([{ "Area":"Holdout isolation","Status":"PASS","Evidence":"2024–2025 not loaded"},{"Area":"Sequence leakage","Status":"PASS","Evidence":"Sequences end at forecast origin"},{"Area":"Target overlap","Status":"PASS","Evidence":"Training target dates resolve before validation starts"},{"Area":"Scaling","Status":"PASS","Evidence":"Scalers fitted on training only"},{"Area":"Model selection","Status":"PASS","Evidence":"LSTM/GRU configuration selected on previous year"},{"Area":"Script09 safeguard","Status":"PASS","Evidence":"DL blend weight selected from previous outer-year predictions"}]); display(quality_gate); quality_gate.to_csv(TAB/"11_deep_learning_quality_gate.csv",index=False); print("SCRIPT 11 COMPLETE — 2024–2025 untouched")